In [ ]:
# Fast & Simple Approach for Maximum Performance
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import torch

# Simple config
MODEL_NAME = 'microsoft/deberta-v3-base'
BATCH_SIZE = 16
EPOCHS = 1
LEARNING_RATE = 2e-5

# Load data
train_df = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
test_df = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')

# Remove data leakage
test_texts = set(test_df['text'].values)
train_df = train_df[~train_df['text'].isin(test_texts)].copy()
print(f"Training size after removing overlap: {len(train_df)}")

# preprocessing
def preprocess(text, keyword=None, location=None):
    # Just combine everything cleanly
    parts = []
    if pd.notna(keyword):
        parts.append(str(keyword))
    if pd.notna(location):
        parts.append(str(location))
    parts.append(str(text))
    return ' '.join(parts)

train_df['input'] = train_df.apply(lambda x: preprocess(x['text'], x['keyword'], x['location']), axis=1)
test_df['input'] = test_df.apply(lambda x: preprocess(x['text'], x['keyword'], x['location']), axis=1)

# SIMPLE DATA AUGMENTATION - Just duplicate disaster examples (they're underrepresented)
disaster_examples = train_df[train_df['target'] == 1].copy()
print(f"Original disasters: {len(disaster_examples)}")
print(f"Original non-disasters: {len(train_df[train_df['target'] == 0])}")

# Add duplicates of disaster examples to balance classes
train_df = pd.concat([train_df, disaster_examples], ignore_index=True)
print(f"After augmentation: {len(train_df)} total examples")
print(f"New class distribution: {train_df['target'].value_counts()}")

# Create datasets
train_ds = Dataset.from_pandas(train_df)
test_ds = Dataset.from_pandas(test_df)

# Tokenization
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize(examples):
    return tokenizer(examples["input"], truncation=True, padding=True, max_length=128)

train_tokenized = train_ds.map(tokenize, batched=True)
test_tokenized = test_ds.map(tokenize, batched=True)
train_tokenized = train_tokenized.rename_column('target', 'labels')

# Simple train/val split
train_indices, val_indices = train_test_split(
    range(len(train_tokenized)), test_size=0.1, random_state=42, 
    stratify=train_tokenized['labels']
)
train_dataset = train_tokenized.select(train_indices)
val_dataset = train_tokenized.select(val_indices)

# CLASS WEIGHTS
class_weights = compute_class_weight('balanced', classes=[0,1], y=train_df['target'])
weight_tensor = torch.tensor(class_weights, dtype=torch.float32)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = torch.nn.CrossEntropyLoss(weight=weight_tensor.to(labels.device))
        loss = loss_fct(logits.view(-1, 2), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# Training setup
args = TrainingArguments(
    output_dir='outputs',
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE*2,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    eval_strategy="no",  # Skip evaluation to save time
    fp16=True,
    report_to='none',
    save_strategy="no",  # Don't save to speed up
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=-1)
    return {'f1': f1_score(labels, predictions)}

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
trainer = WeightedTrainer(model=model, args=args, train_dataset=train_dataset, 
                         eval_dataset=val_dataset, tokenizer=tokenizer, compute_metrics=compute_metrics)

# Train
print("Training...")
trainer.train()

# Quick validation check
val_results = trainer.evaluate()
print(f"Validation F1: {val_results['eval_f1']:.4f}")

# PSEUDO-LABELING - Use high confidence predictions as additional training data
print("Getting test predictions for pseudo-labeling...")
test_pred = trainer.predict(test_tokenized)
test_probs = torch.softmax(torch.tensor(test_pred.predictions), dim=-1).numpy()
test_confidence = np.max(test_probs, axis=-1)
test_pred_classes = np.argmax(test_pred.predictions, axis=-1)

# Use high confidence predictions (>90%) as additional training data
high_conf_mask = test_confidence > 0.9
high_conf_examples = test_df[high_conf_mask].copy()
high_conf_examples['target'] = test_pred_classes[high_conf_mask]

print(f"High confidence pseudo-labels: {len(high_conf_examples)} ({len(high_conf_examples)/len(test_df):.2%})")
print(f"Pseudo-label distribution: {np.bincount(high_conf_examples['target'])}")

# Add pseudo-labeled data to training set
extended_train_df = pd.concat([train_df, high_conf_examples[['input', 'target']]], ignore_index=True)
print(f"Extended training size: {len(extended_train_df)}")

# Retrain with pseudo-labeled data
extended_ds = Dataset.from_pandas(extended_train_df)
extended_tokenized = extended_ds.map(tokenize, batched=True)
extended_tokenized = extended_tokenized.rename_column('target', 'labels')

# Update class weights for extended dataset
new_class_weights = compute_class_weight('balanced', classes=[0,1], y=extended_train_df['target'])
weight_tensor = torch.tensor(new_class_weights, dtype=torch.float32)

# Create new trainer with extended data
extended_trainer = WeightedTrainer(
    model=model, args=args, train_dataset=extended_tokenized, 
    eval_dataset=val_dataset, tokenizer=tokenizer, compute_metrics=compute_metrics
)

print("Retraining with pseudo-labeled data...")
extended_trainer.train()

# Final predictions
final_pred = extended_trainer.predict(test_tokenized)
final_classes = np.argmax(final_pred.predictions, axis=-1)

# Create submission
submission = pd.read_csv('/kaggle/input/nlp-getting-started/sample_submission.csv')
submission['target'] = final_classes
submission.to_csv('submission_fast.csv', index=False)

print(f"Final prediction distribution: {np.bincount(final_classes)}")
print(f"Submission created!")

# Quick analysis
final_val = extended_trainer.evaluate()
print(f"Final validation F1: {final_val['eval_f1']:.4f}")